# UAS AI FOR DATA SCIENTIST — UNIVERSITAS ISLAM INDONESIA
## Sistem Deteksi Kendaraan — YouTube Live CCTV (v2: Akurasi Ditingkatkan)

---

### Perbaikan v2
| Masalah | Solusi |
|---|---|
| Truk dikira Bus | Refinement pasca-YOLO berdasarkan aspect ratio bbox |
| Motor dikira Mobil | Threshold area & ukuran dioptimalkan untuk lalu lintas Indonesia |
| Mobil banyak tidak terhitung | max_distance 120px + velocity prediction + cooldown anti double-count |
| Deteksi duplikat | NMS (Non-Maximum Suppression) ditambahkan |

---

**Cara Menjalankan:**  Jalankan semua sel → di sel **PERCOBAAN 3** tekan Run → tekan **Stop** untuk menghentikan.

## TAHAP 0 — Instalasi Library

In [ ]:
import sys
!{sys.executable} -m pip install ultralytics opencv-python-headless numpy pandas matplotlib yt-dlp --quiet
print('Instalasi selesai!')

## TAHAP 1 — Import Library

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import os
import time
import subprocess
import math
from IPython.display import display, HTML, clear_output, Image as IPImage
from PIL import Image as PILImage

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
    print('Ultralytics YOLO OK')
except ImportError:
    YOLO_AVAILABLE = False
    print('YOLO tidak tersedia, pakai fallback kontur')

print('OpenCV :', cv2.__version__)
print('NumPy  :', np.__version__)

## TAHAP 2 — Konfigurasi

In [ ]:
VEHICLE_CLASS_MAP = {2: 'Mobil', 3: 'Sepeda Motor', 5: 'Bus', 7: 'Truk'}

CLASS_COLORS = {
    'Mobil':        (0, 215, 255),
    'Sepeda Motor': (255, 144, 30),
    'Bus':          (50, 205, 50),
    'Truk':         (50, 50, 240)
}

CHART_COLORS = {
    'Mobil': '#FFD700', 'Sepeda Motor': '#1E90FF',
    'Bus': '#32CD32',   'Truk': '#FF4500'
}

TARGET_CLASSES   = [2, 3, 5, 7]
YOUTUBE_LIVE_URL = 'https://www.youtube.com/live/06Xji1C_5Ak?si=Vk7rwbgJ1mtwl_Ck'
os.makedirs('output', exist_ok=True)

print('Konfigurasi OK')
print('URL:', YOUTUBE_LIVE_URL)

## TAHAP 3 — Ekstrak URL Stream dari YouTube Live

In [ ]:
def get_stream_url(youtube_url):
    print('Mengekstrak stream URL...')
    for fmt in ['best[height<=720][ext=mp4]', 'best[height<=480]', 'best']:
        try:
            r = subprocess.run(
                ['yt-dlp', '--no-warnings', '-f', fmt, '-g', youtube_url],
                capture_output=True, text=True, timeout=60)
            if r.returncode == 0 and r.stdout.strip():
                url = r.stdout.strip().split('\n')[0]
                print('OK! Format:', fmt)
                print('URL:', url[:80], '...')
                return url
        except Exception as e:
            print('Error:', e)
    print('GAGAL mendapatkan stream URL.')
    return None


STREAM_URL = get_stream_url(YOUTUBE_LIVE_URL)
print('Stream tersedia:', STREAM_URL is not None)

## TAHAP 4 — VehicleDetector v2

Perbaikan:
- **`_refine_class()`**: koreksi Truk vs Bus dan Motor vs Mobil berdasarkan ukuran bbox
- **NMS**: hapus deteksi duplikat/overlapping
- **conf=0.25**: lebih rendah = lebih banyak kendaraan terdeteksi

In [ ]:
def _refine_class(cls_id, cls_name, bbox, conf):
    """
    Koreksi klasifikasi YOLO berdasarkan ukuran bounding box.
    Mengatasi kesalahan Truk/Bus dan Motor/Mobil di lalu lintas Indonesia.
    """
    x1, y1, x2, y2 = bbox
    w    = max(1, x2 - x1)
    h    = max(1, y2 - y1)
    area = w * h
    ar   = w / h

    if cls_name in ('Bus', 'Truk'):
        if ar > 2.5:
            return 'Bus', 5, conf
        elif ar < 1.55:
            return 'Truk', 7, conf
        else:
            return ('Truk', 7, conf) if h > w * 0.62 else ('Bus', 5, conf)

    if cls_name == 'Mobil':
        if area < 2800 and w < 75:
            return 'Sepeda Motor', 3, conf
        if area < 4500 and w < 85 and h > w * 0.82:
            return 'Sepeda Motor', 3, conf

    if cls_name == 'Sepeda Motor':
        if area > 18000 or (w > 160 and h > 80):
            return 'Mobil', 2, conf

    return cls_name, cls_id, conf


def _nms(detections, iou_thr=0.45):
    if len(detections) <= 1:
        return detections
    boxes  = np.array([d['bbox'] for d in detections], dtype=float)
    scores = np.array([d['confidence'] for d in detections])
    x1,y1,x2,y2 = boxes[:,0],boxes[:,1],boxes[:,2],boxes[:,3]
    areas  = (x2-x1+1)*(y2-y1+1)
    order  = scores.argsort()[::-1]
    keep   = []
    while order.size > 0:
        i = order[0]; keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0,xx2-xx1+1)*np.maximum(0,yy2-yy1+1)
        iou   = inter/(areas[i]+areas[order[1:]]-inter)
        order = order[np.where(iou<=iou_thr)[0]+1]
    return [detections[i] for i in keep]


class VehicleDetector:
    """
    YOLOv8 detector dengan refinement klasifikasi dan NMS.
    conf=0.25 untuk recall tinggi, dilengkapi koreksi Truk/Bus & Motor/Mobil.
    """

    def __init__(self, model_name='yolov8n.pt', conf=0.25, iou=0.45):
        self.conf  = conf
        self.iou   = iou
        self.model = None
        if YOLO_AVAILABLE:
            try:
                self.model = YOLO(model_name)
                print('Model', model_name, 'siap. conf='+str(conf))
            except Exception as e:
                print('Gagal memuat YOLO:', e)
        else:
            print('Pakai kontur fallback.')

    def detect(self, frame):
        if frame is None or frame.size == 0:
            return []
        dets = []
        if self.model:
            res = self.model.predict(
                source=frame, conf=self.conf, iou=self.iou,
                classes=TARGET_CLASSES, verbose=False, agnostic_nms=True)
            if res and len(res[0].boxes) > 0:
                for b in res[0].boxes:
                    x1,y1,x2,y2 = b.xyxy[0].cpu().numpy().astype(int)
                    if (x2-x1) < 18 or (y2-y1) < 14:
                        continue
                    cid  = int(b.cls[0])
                    name = VEHICLE_CLASS_MAP.get(cid, 'Kendaraan')
                    conf_val = float(b.conf[0])
                    name, cid, conf_val = _refine_class(cid, name, (x1,y1,x2,y2), conf_val)
                    dets.append({'bbox':(x1,y1,x2,y2), 'class_name':name,
                                 'confidence':conf_val, 'class_id':cid})
        if not dets:
            dets = self._contour(frame)
        else:
            dets = _nms(dets, self.iou)
        return dets

    def _contour(self, frame):
        """Fallback deteksi kontur dioptimalkan untuk lalu lintas Indonesia."""
        dets = []
        h, w = frame.shape[:2]
        t, b = int(h*0.18), int(h*0.88)
        roi  = frame[t:b, :]
        gray = cv2.GaussianBlur(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), (7,7), 0)
        edges = cv2.Canny(gray, 30, 100)
        _, thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        mask = cv2.bitwise_or(edges, thr)
        kc = cv2.getStructuringElement(cv2.MORPH_RECT,(7,7))
        ko = cv2.getStructuringElement(cv2.MORPH_RECT,(4,4))
        mask = cv2.morphologyEx(cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kc),cv2.MORPH_OPEN,ko)

        for cnt in cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
            area = cv2.contourArea(cnt)
            if area < 1200:
                continue
            x, y, bw, bh = cv2.boundingRect(cnt)
            ay   = y + t
            ar   = bw / max(1, bh)
            fill = area / max(1, bw*bh)
            if bw > w*0.70 or bh < 12 or ay < 30 or fill < 0.15:
                continue

            if area > 22000 or bw > 175:
                c = ('Bus',5,0.82) if ar > 2.4 else ('Truk',7,0.80)
            elif area < 5000 and bw < 90 and bh < 75:
                c = ('Sepeda Motor',3,0.78)
            elif area < 3500 and bw < 70:
                c = ('Sepeda Motor',3,0.75)
            elif 5000 <= area <= 22000 and bw >= 70:
                c = ('Mobil',2,0.80)
            else:
                c = ('Mobil',2,0.70)

            dets.append({'bbox':(x,ay,x+bw,ay+bh),
                         'class_name':c[0],'confidence':c[2],'class_id':c[1]})

        return _nms(dets, 0.40)

    def draw(self, frame, dets):
        out = frame.copy()
        for d in dets:
            x1,y1,x2,y2 = d['bbox']
            cls  = d['class_name']
            conf = d['confidence']
            col  = CLASS_COLORS.get(cls,(0,255,0))
            cv2.rectangle(out,(x1,y1),(x2,y2),col,2)
            ll = min(14,(x2-x1)//4,(y2-y1)//4)
            if ll > 3:
                for p0,p1 in [((x1,y1),(x1+ll,y1)),((x1,y1),(x1,y1+ll)),
                              ((x2,y1),(x2-ll,y1)),((x2,y1),(x2,y1+ll))]:
                    cv2.line(out,p0,p1,col,3)
            lbl = cls+' '+str(round(conf*100))+'%'
            (tw,th),_ = cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.50,1)
            cv2.rectangle(out,(x1,max(0,y1-th-7)),(x1+tw+8,y1),col,-1)
            cv2.putText(out,lbl,(x1+4,y1-3),
                        cv2.FONT_HERSHEY_SIMPLEX,0.50,(10,10,10),1,cv2.LINE_AA)
        return out


detector = VehicleDetector('yolov8n.pt', conf=0.25)
print('VehicleDetector v2 siap.')

## TAHAP 5 — VehicleTracker v2

Perbaikan:
- **max_distance=120px** (dari 50px) — mengikuti kendaraan yang bergerak cepat
- **Velocity prediction** — prediksi posisi berdasarkan kecepatan sebelumnya
- **Cooldown anti-double-count** — mencegah satu kendaraan dihitung dua kali
- **Garis ROI horizontal** — lebih efektif untuk CCTV overhead/samping jalan

In [ ]:
class VehicleTracker:
    """
    Tracker dengan velocity prediction dan cooldown anti-double-count.
    max_distance=120px untuk mengikuti kendaraan yang bergerak cepat.
    """

    def __init__(self, max_dist=120, max_gone=25):
        self.next_id    = 1
        self.objects    = {}; self.vels = {}; self.data = {}; self.gone = {}
        self.max_dist   = max_dist
        self.max_gone   = max_gone
        self.counts     = {'Mobil':0,'Sepeda Motor':0,'Bus':0,'Truk':0}
        self.history    = []

    def _reg(self, cen, det):
        oid = self.next_id
        self.objects[oid]  = cen
        self.vels[oid]     = (0.0, 0.0)
        self.data[oid]     = {'class_name': det['class_name'],
                              'counted': False, 'cooldown': 0, 'hist': [cen]}
        self.gone[oid]     = 0
        self.next_id      += 1

    def _del(self, oid):
        for d in [self.objects, self.vels, self.data, self.gone]:
            d.pop(oid, None)

    def update(self, dets, line=None):
        tracked = []

        for oid in self.data:
            if self.data[oid]['cooldown'] > 0:
                self.data[oid]['cooldown'] -= 1

        if not dets:
            for oid in list(self.gone):
                self.gone[oid] += 1
                if self.gone[oid] > self.max_gone:
                    self._del(oid)
            return tracked

        inp = [((d['bbox'][0]+d['bbox'][2])//2,
                (d['bbox'][1]+d['bbox'][3])//2) for d in dets]

        if not self.objects:
            for i, d in enumerate(dets):
                self._reg(inp[i], d)
                dc = d.copy(); dc['object_id'] = self.next_id-1
                tracked.append(dc)
            return tracked

        oids = list(self.objects)
        pred = [
            (int(self.objects[oid][0] + self.vels[oid][0]),
             int(self.objects[oid][1] + self.vels[oid][1]))
            for oid in oids
        ]

        D  = np.linalg.norm(np.array(pred)[:,None]-np.array(inp)[None,:],axis=2)
        rows = D.min(axis=1).argsort()
        cols = D.argmin(axis=1)[rows]
        ur, uc = set(), set()

        for r, c in zip(rows, cols):
            if r in ur or c in uc or D[r,c] > self.max_dist:
                continue
            oid  = oids[r]
            prev = self.objects[oid]
            curr = inp[c]

            vx = 0.6*(curr[0]-prev[0]) + 0.4*self.vels[oid][0]
            vy = 0.6*(curr[1]-prev[1]) + 0.4*self.vels[oid][1]
            self.vels[oid]    = (vx, vy)
            self.objects[oid] = curr
            self.gone[oid]    = 0
            self.data[oid]['class_name'] = dets[c]['class_name']

            hist = self.data[oid]['hist']
            hist.append(curr)
            if len(hist) > 30:
                hist.pop(0)

            if line:
                self._check(oid, prev, curr, line)

            dc = dets[c].copy(); dc['object_id'] = oid
            tracked.append(dc)
            ur.add(r); uc.add(c)

        for r, oid in enumerate(oids):
            if r not in ur:
                self.gone[oid] += 1
                if self.gone[oid] > self.max_gone:
                    self._del(oid)

        for c in range(len(inp)):
            if c not in uc:
                self._reg(inp[c], dets[c])
                dc = dets[c].copy(); dc['object_id'] = self.next_id-1
                tracked.append(dc)

        return tracked

    def _check(self, oid, p1, p2, line):
        """Cek apakah kendaraan melintas garis ROI (vertikal atau horizontal)."""
        if self.data[oid]['counted'] or self.data[oid]['cooldown'] > 0:
            return
        l1, l2 = line
        crossed = False

        if l1[0] == l2[0]:
            lx = l1[0]
            miny, maxy = min(l1[1],l2[1]), max(l1[1],l2[1])
            in_y = (miny <= p2[1] <= maxy) or (miny <= p1[1] <= maxy)
            crossed = in_y and ((p1[0]<lx and p2[0]>=lx) or (p1[0]>lx and p2[0]<=lx))

        elif l1[1] == l2[1]:
            ly = l1[1]
            minx, maxx = min(l1[0],l2[0]), max(l1[0],l2[0])
            in_x = (minx <= p2[0] <= maxx) or (minx <= p1[0] <= maxx)
            crossed = in_x and ((p1[1]<ly and p2[1]>=ly) or (p1[1]>ly and p2[1]<=ly))

        if crossed:
            cls = self.data[oid]['class_name']
            if cls in self.counts:
                self.counts[cls] += 1
            self.data[oid]['counted']  = True
            self.data[oid]['cooldown'] = 15
            self.history.append({'oid': oid, 'cls': cls, 'n': len(self.history)+1})

    def draw_hud(self, frame, line=None, fps=0.0):
        out  = frame.copy()
        h, w = out.shape[:2]

        if line:
            cv2.line(out, line[0], line[1], (0,255,255), 2, cv2.LINE_AA)
            cv2.putText(out,'ROI COUNTER',
                        (line[0][0]+8, line[0][1]-8 if line[0][1]>30 else line[0][1]+20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.44, (0,255,255), 1, cv2.LINE_AA)

        panel_w, panel_h = 295, 168
        ov = out.copy()
        cv2.rectangle(ov,(8,8),(8+panel_w,8+panel_h),(5,10,20),-1)
        cv2.addWeighted(ov,0.70,out,0.30,0,out)
        cv2.rectangle(out,(8,8),(8+panel_w,8+panel_h),(0,215,255),1)

        total = sum(self.counts.values())
        cv2.putText(out,'UII ITS AI COUNTER',(16,28),
                    cv2.FONT_HERSHEY_SIMPLEX,0.46,(0,215,255),1,cv2.LINE_AA)
        cv2.putText(out,'FPS:'+str(round(fps,1)),(215,28),
                    cv2.FONT_HERSHEY_SIMPLEX,0.42,(180,180,180),1,cv2.LINE_AA)

        y = 52
        cv2.putText(out,'Total: '+str(total),(16,y),
                    cv2.FONT_HERSHEY_SIMPLEX,0.52,(255,255,255),1,cv2.LINE_AA)
        y += 22
        for key, lbl, col in [
            ('Mobil','Mobil',(0,215,255)),
            ('Sepeda Motor','Motor',(255,144,30)),
            ('Bus','Bus',(50,205,50)),
            ('Truk','Truk',(80,80,255))
        ]:
            cv2.putText(out,'  '+lbl+': '+str(self.counts.get(key,0)),
                        (16,y),cv2.FONT_HERSHEY_SIMPLEX,0.44,col,1,cv2.LINE_AA)
            y += 20

        cv2.putText(out,'FPS '+str(round(fps,1)),(w-95,30),
                    cv2.FONT_HERSHEY_SIMPLEX,0.54,(0,255,100),1,cv2.LINE_AA)
        return out


print('VehicleTracker v2 siap.')

## TAHAP 6 — Helper: Tampilkan Frame di Jupyter

In [ ]:
def to_ipimage(frame, width=860):
    h, w = frame.shape[:2]
    nh = int(h * width / w)
    rsz = cv2.resize(frame, (width, nh))
    rgb = cv2.cvtColor(rsz, cv2.COLOR_BGR2RGB)
    buf = io.BytesIO()
    PILImage.fromarray(rgb).save(buf, format='JPEG', quality=85)
    buf.seek(0)
    return IPImage(data=buf.read())


def status_bar(frame, text):
    h, w = frame.shape[:2]
    cv2.rectangle(frame,(0,h-30),(w,h),(10,10,10),-1)
    cv2.putText(frame,text,(8,h-9),cv2.FONT_HERSHEY_SIMPLEX,0.52,(0,255,200),1,cv2.LINE_AA)
    return frame


print('Helper siap.')

---
# PERCOBAAN 1 — Tes Koneksi: Ambil 1 Frame

In [ ]:
def grab_frame(url, skip=8):
    if not url:
        print('Stream URL tidak ada.')
        return None
    cap = cv2.VideoCapture(url)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 2)
    if not cap.isOpened():
        print('Gagal membuka stream.')
        cap.release(); return None
    for _ in range(skip):
        cap.grab()
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print('Gagal baca frame.')
        return None
    print('Frame OK:', frame.shape[1], 'x', frame.shape[0])
    return frame


raw_frame = grab_frame(STREAM_URL)
if raw_frame is not None:
    cv2.imwrite('output/p1_raw.jpg', raw_frame)
    display(HTML('<b>Frame Mentah dari YouTube Live:</b>'))
    display(to_ipimage(raw_frame))
else:
    print('Cek STREAM_URL di Tahap 3.')

---
# PERCOBAAN 2 — Deteksi Single-Shot + Tabel Refinement

In [ ]:
if raw_frame is not None:
    t0   = time.time()
    dets = detector.detect(raw_frame)
    ann  = detector.draw(raw_frame, dets)
    ms   = (time.time()-t0)*1000

    ann = status_bar(ann, str(len(dets))+' kendaraan | '+str(round(ms))+' ms | '
                    +('YOLOv8+Refinement' if detector.model else 'Kontur'))
    cv2.imwrite('output/p2_detected.jpg', ann)
    display(HTML('<b>Hasil Deteksi Single-Shot (v2):</b>'))
    display(to_ipimage(ann))

    if dets:
        df = pd.DataFrame([{
            'No': i+1,
            'Kelas': d['class_name'],
            'Conf':  str(round(d['confidence']*100))+'%',
            'W':     d['bbox'][2]-d['bbox'][0],
            'H':     d['bbox'][3]-d['bbox'][1],
            'AR':    str(round((d['bbox'][2]-d['bbox'][0])/max(1,d['bbox'][3]-d['bbox'][1]),2))
        } for i,d in enumerate(dets)])
        display(HTML('<h4>Detail Deteksi (W=Lebar, H=Tinggi, AR=Aspect Ratio)</h4>'))
        display(df.style.hide(axis='index')
            .set_table_styles([{'selector':'th','props':[('background','#1a3a5c'),('color','white')]}]))
    print('Total:', len(dets), '| Waktu:', round(ms,1), 'ms')
else:
    print('Lewati — frame tidak tersedia.')

---
# PERCOBAAN 3 — LIVE STREAM REAL-TIME (Berjalan Terus)

> **Sel ini menjalankan deteksi secara LIVE dan terus-menerus.**
>
> Garis ROI **HORIZONTAL** di tengah frame (lebih efektif untuk menghitung kendaraan
> yang bergerak vertikal di layar, seperti pada CCTV tepi jalan).
>
> Tekan **tombol Stop (kotak hitam)** untuk menghentikan.
> Tekan **I kemudian I lagi** untuk interrupt kernel Jupyter.

In [ ]:
def run_live(url, width=860, snap_every=120, snap_path='output/live_snap.jpg'):
    """
    LIVE stream deteksi kendaraan dari YouTube.
    Frame di-render real-time di Jupyter dengan clear_output.
    Berjalan sampai ditekan Stop.

    ROI Line: HORIZONTAL di 55% tinggi frame (efektif untuk kendaraan bergerak vertikal).
    """
    if not url:
        display(HTML('<span style="color:red">Stream URL tidak ada. Jalankan Tahap 3.</span>'))
        return None

    trk = VehicleTracker(max_dist=120, max_gone=25)
    cap = cv2.VideoCapture(url)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    if not cap.isOpened():
        display(HTML('<span style="color:red">Gagal membuka stream.</span>'))
        return None

    W       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or 1280
    H       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720

    # ROI Line HORIZONTAL di 55% tinggi frame
    # Sesuaikan nilai ini jika ingin garis di posisi berbeda
    roi_y    = int(H * 0.55)
    roi_line = ((int(W*0.05), roi_y), (int(W*0.95), roi_y))

    display(HTML(
        '<div style="background:#0d1117;padding:10px;border-radius:8px;color:white;font-family:monospace;">'
        '<b style="color:#00FFCC">LIVE DETECTION v2 AKTIF</b> — '
        'Resolusi: '+str(W)+'x'+str(H)+' | '
        'ROI Line: y='+str(roi_y)+' (horizontal)<br>'
        '<span style="color:#FFD700">Perbaikan: Truk/Bus akurat, Motor/Mobil akurat, '
        'counting lebih lengkap</span><br>'
        '<span style="color:#aaa">Tekan STOP untuk menghentikan.</span>'
        '</div>'
    ))

    fidx     = 0
    fps_buf  = []
    t_start  = time.time()

    try:
        while True:
            cap.grab()            # Buang frame buffer lama
            ret, frame = cap.read()

            if not ret or frame is None:
                display(HTML('<span style="color:orange">Stream putus, reconnect...</span>'))
                time.sleep(2)
                cap.release()
                cap = cv2.VideoCapture(url)
                cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
                continue

            fidx += 1
            t0    = time.time()

            dets    = detector.detect(frame)
            tracked = trk.update(dets, line=roi_line)
            ann     = detector.draw(frame, tracked)
            fps     = 1.0 / max(0.001, time.time()-t0)
            ann     = trk.draw_hud(ann, roi_line, fps)

            fps_buf.append(fps)
            if len(fps_buf) > 30:
                fps_buf.pop(0)
            avg_fps = sum(fps_buf)/len(fps_buf)
            elapsed = int(time.time()-t_start)
            total   = sum(trk.counts.values())

            bar = ('Frame:'+str(fidx)+
                   ' | Det:'+str(len(dets))+
                   ' | FPS:'+str(round(avg_fps,1))+
                   ' | Melintas:'+str(total)+
                   ' | '+str(elapsed)+'s')
            ann = status_bar(ann, bar)

            if fidx % snap_every == 0:
                cv2.imwrite(snap_path, ann)

            clear_output(wait=True)
            display(to_ipimage(ann, width=width))
            display(HTML(
                '<div style="background:#0d1117;padding:8px;border-radius:6px;'
                'font-family:monospace;font-size:13px;">'
                '<span style="color:#00FFCC">LIVE</span> '
                '<span style="color:#FFD700">Frame '+str(fidx)+'</span> | '
                '<span style="color:#00FF88">FPS: '+str(round(avg_fps,1))+'</span> | '
                '<span style="color:white">'+str(elapsed)+' detik</span><br>'
                '<span style="color:#FFD700">Mobil: '+str(trk.counts['Mobil'])+'</span>  '
                '<span style="color:#1E90FF">Motor: '+str(trk.counts['Sepeda Motor'])+'</span>  '
                '<span style="color:#32CD32">Bus: '+str(trk.counts['Bus'])+'</span>  '
                '<span style="color:#FF4500">Truk: '+str(trk.counts['Truk'])+'</span>  '
                '<b style="color:white">| TOTAL: '+str(total)+'</b>'
                '</div>'
            ))

    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
        elapsed_total = int(time.time()-t_start)

    avg_final = sum(fps_buf)/max(1,len(fps_buf))
    clear_output(wait=True)
    total = sum(trk.counts.values())
    display(HTML(
        '<div style="background:#1a3a2e;border:1px solid #32CD32;padding:14px;border-radius:10px;">'
        '<h3 style="color:#00FFCC;margin:0 0 10px">LIVE STREAM SELESAI</h3>'
        '<table style="color:white;width:100%;border-collapse:collapse;">'
        '<tr><td>Total Frame</td><td><b>'+str(fidx)+'</b></td></tr>'
        '<tr><td>Durasi</td><td><b>'+str(elapsed_total)+' detik</b></td></tr>'
        '<tr><td>Rata-rata FPS</td><td><b>'+str(round(avg_final,1))+'</b></td></tr>'
        '<tr><td style="color:#FFD700">Mobil</td><td><b style="color:#FFD700">'+str(trk.counts['Mobil'])+'</b></td></tr>'
        '<tr><td style="color:#1E90FF">Sepeda Motor</td><td><b style="color:#1E90FF">'+str(trk.counts['Sepeda Motor'])+'</b></td></tr>'
        '<tr><td style="color:#32CD32">Bus</td><td><b style="color:#32CD32">'+str(trk.counts['Bus'])+'</b></td></tr>'
        '<tr><td style="color:#FF4500">Truk</td><td><b style="color:#FF4500">'+str(trk.counts['Truk'])+'</b></td></tr>'
        '<tr style="border-top:1px solid #555"><td><b>TOTAL MELINTAS</b></td>'
        '<td><b style="color:#00FFCC;font-size:20px">'+str(total)+'</b></td></tr>'
        '</table></div>'
    ))
    return trk


live_tracker = run_live(
    url        = STREAM_URL,
    width      = 860,
    snap_every = 120,
    snap_path  = 'output/live_snap.jpg'
)

---
# PERCOBAAN 4 — Grafik Distribusi (Setelah Live Stream Dihentikan)

In [ ]:
if live_tracker and sum(live_tracker.counts.values()) > 0:
    counts = live_tracker.counts
    total  = sum(counts.values())
    labels = [k for k,v in counts.items() if v > 0]
    values = [v for v in counts.values() if v > 0]
    colors = [CHART_COLORS[l] for l in labels]

    fig, axes = plt.subplots(1, 2, figsize=(13,5))
    fig.patch.set_facecolor('#1a1a2e')
    fig.suptitle('Distribusi Kendaraan — YouTube Live CCTV',
                 color='white', fontsize=13, fontweight='bold')

    ax1 = axes[0]; ax1.set_facecolor('#16213e')
    bars = ax1.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
    ax1.set_title('Jumlah Kendaraan Melintas ROI', color='white')
    ax1.set_xlabel('Jenis', color='#aaa'); ax1.set_ylabel('Unit', color='#aaa')
    ax1.tick_params(colors='white')
    for bar, val in zip(bars, values):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 str(val), ha='center', color='white', fontweight='bold', fontsize=12)

    ax2 = axes[1]; ax2.set_facecolor('#16213e')
    wedges,texts,auts = ax2.pie(values, labels=labels, colors=colors, autopct='%1.1f%%',
                                startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
    for t in texts:   t.set_color('white')
    for at in auts:   at.set_color('white'); at.set_fontweight('bold')
    ax2.set_title('Proporsi (%)', color='white')

    plt.tight_layout()
    plt.savefig('output/distribusi.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    print('Grafik: output/distribusi.png')

    df = pd.DataFrame([{'Kelas':k,'Jumlah':v,'Persen':str(round(v/max(1,total)*100,1))+'%'}
                        for k,v in counts.items()])
    df.loc[len(df)] = ['TOTAL', total, '100%']
    display(df.style.hide(axis='index')
        .set_table_styles([{'selector':'th','props':[('background','#1a3a5c'),('color','white')]}]))
else:
    print('Jalankan Live Stream (Percobaan 3) terlebih dahulu.')

---
# Lanjut ke Streamlit Dashboard

In [ ]:
display(HTML(
    '<div style="background:#0f2027;border:1px solid #00d7ff;padding:18px;border-radius:12px;">'
    '<h3 style="color:#00d7ff;margin:0 0 10px">Streamlit Dashboard</h3>'
    '<p style="color:white">1. Buka Terminal baru</p>'
    '<p style="color:#aaa"><code style="background:#222;padding:4px 8px;border-radius:4px">'
    'cd "d:\\AI for data scientist\\UASAI"</code></p>'
    '<p style="color:#aaa"><code style="background:#222;padding:4px 8px;border-radius:4px">'
    'streamlit run app.py</code></p>'
    '<p style="color:#FFD700">Browser: http://localhost:8501</p>'
    '<p style="color:#aaa">Sidebar: pilih YouTube Live Stream → klik Ambil Stream URL</p>'
    '<hr style="border-color:#333">'
    '<b style="color:#00FFCC">URL YouTube Live:</b><br>'
    '<span style="color:white">'+YOUTUBE_LIVE_URL+'</span>'
    '</div>'
))

files = [
    ('output/p1_raw.jpg','Frame mentah'),
    ('output/p2_detected.jpg','Single-shot detection'),
    ('output/live_snap.jpg','Snapshot live stream'),
    ('output/distribusi.png','Grafik distribusi'),
]
rows = []
for p, d in files:
    ex = os.path.exists(p)
    sz = os.path.getsize(p)/1024 if ex else 0
    rows.append({'File':p,'Deskripsi':d,'Status':'Ada' if ex else 'Belum',
                 'Ukuran':str(round(sz,1))+' KB' if ex else '-'})
display(HTML('<h4>Status Output</h4>'))
display(pd.DataFrame(rows).style.hide(axis='index')
    .set_table_styles([{'selector':'th','props':[('background','#1a3a5c'),('color','white')]}]))